In [1]:
%pip install PyPDF2 opensearch-py sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import re
import os
import sys
from typing import Dict, List, Any

from PyPDF2 import PdfReader
from opensearchpy import OpenSearch, helpers
from sentence_transformers import SentenceTransformer

c:\Users\ksuma\OneDrive\Documents\local-rag-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
OPENSEARCH_HOST = "localhost"
OPENSEARCH_PORT = 9200
OPENSEARCH_INDEX =" documents"

EMBEDDING_MODEL = " sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIMENSIONS = 384
ASYMETRIC_EMBEDDING = False

TEXT_CHUNK_SIZE = 500
TEXT_CHUNK_OVERLAP = 100


In [5]:
def clean_text(text: str) -> str:
    """Cleans OCR-extracted text by removing unnecessary newlines, hyphens, and correcting common OCR errors.

    Args:
        text (str): The text to clean.

    Returns:
        str: The cleaned text."""

    text = re.sub(r"(\w+)-\n(\w+)", r"\1\2", text)
    text = re.sub(r"(?<!\n)\n(?!\n)", text)
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [6]:
def chunk_text(text: str, chunk_size: int, overlap: int = 100) -> List[str]:
    """Splits text into chunks with a specified overlap.

    Args:
        text (str): The text to split.
        chunk_size (int): The number of tokens in each chunk.
        overlap (int): The number of tokens to overlap between chunks.

    Returns:
        List[str]: A list of text chunks."""

    text = clean_text(text)
    tokens = text.split(" ")
    chunks = []
    start = 0
    while start <= len(tokens):
        end = start + chunk_size
        chunk_token = tokens[start:end]
        chunk_text = " ".join(chunk_token)
        chunks.append(chunk_text)
        start = end - overlap
    
    return chunks

In [7]:
def get_embedding_model():
    """Loads and returns the sentence transformer embedding model.
    
    Returns:
        SentenceTransformer: The loaded embedding model."""
    model = SentenceTransformer(EMBEDDING_MODEL)
    return model

def generate_embedding(texts: List[str]):
    """
    Generates embeddings for a list of text chunks.
    
    Args:
        texts (List[str]): List of text chunks to embed.
        
    Returns:
        List[numpy.ndarray]: List of embedding vectors.
    """
    model = get_embedding_model()

    if ASYMETRIC_EMBEDDING :
        texts = [f"passage : {text}" for text in texts]

    embedding = model(texts)
    return embedding
    